# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

D. S  ->  D. S. Anderson  |  ['D. S']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
A. Dumont  ->  A. Dumont  |  ['A. Dumont']
N. Neumayer  ->  N. Neumayer  |  ['N. Neumayer']


P. Smith  ->  P. Smith  |  ['P. Smith']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
N. Wang  ->  N. Wang  |  ['N. Wang']
K. Jahnke  ->  K. Jahnke  |  ['K. Jahnke']


J. Li  ->  J. Li  |  ['J. Li']
A. Dumont  ->  A. Dumont  |  ['A. Dumont']
Arxiv has 93 new papers today
          13 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/13 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.05438


extracting tarball to tmp_2611.05438... done.
Retrieving document from  https://arxiv.org/e-print/2611.05482


extracting tarball to tmp_2611.05482...

 done.


/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.05482/EC_forecasts.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'author_list' from 'tmp_2611.05482/author_list.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))


... (22 outputs left out)

Retrieving document from  https://arxiv.org/e-print/2611.06873
extracting tarball to tmp_2611.06873...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.06916


extracting tarball to tmp_2611.06916... done.


/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Multiple tex files.

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3823: LatexWarning: Found documentclass in tmp_2611.06916/reveallagn_seds.tex

  exec(code_obj, self.user_global_ns, self.user_ns)
/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'silicate_table.tex' from 'tmp_2611.06916/silicate_table.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.17/x64/lib/python3.11/site-packages/arxiv_on_deck_2/latex.py:414: LatexWarning: Latex injecting: 'spec_stub' from 'tmp_2611.06916/spec_stub.tex'
  warnings.warn(LatexWarning(f"Latex injecting: '{ext}' from '{subsource}'"))
/opt/hostedtoolcache/Python/3.11.17/x64/lib/py

### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.05496-b31b1b.svg)](https://arxiv.org/abs/2611.05496) | **ReveaLLAGN 2: Exploring the Nuclear Gas and Stellar Kinematics of Nearby LLAGNs with JWST/NIRSpec IFU**  |
|| <mark>A. Dumont</mark>, et al. -- incl., <mark>N. Neumayer</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *Accepted for publication in Astronomy & Astrophysics*|
|**Abstract**|            Low-luminosity active galactic nuclei (LLAGN) represent the most common mode of black hole accretion in the local Universe, yet their nuclear structure, stellar and gas kinematics remain poorly constrained at parsec scales. We investigate the nuclear regions of eight nearby representative LLAGN, the ReveaLLAGN sample: M87, M104, M94, M81, NGC1052, NGC4258, NGC5128 and NGC4395. Using JWST/NIRSpec data, we probe the stellar, molecular hydrogen, hydrogen recombination, and high ionisation coronal lines kinematics and their spatial distribution, exploring how detectability and spatial distributions of these lines vary across different Eddington ratios. The stellar LOSV maps reveal a slowly rotating disk in the nuclei of all our targets, characterised by an anti-correlation with the h3 Gauss-Hermite moment. Across the sample, the stellar velocity dispersion rises toward the central black hole. Only in M104 and M94 does the velocity dispersion display a more intricate pattern that departs from a simple disk configuration, indicating the presence of multiple kinematic components in these galaxies. Across the sample, the gas distribution and kinematics exhibit rotation that is frequently decoupled from the stellar rotation, suggesting that the stellar gravitational potential is decoupled from the current accretion state. The gas kinematics are often visually aligned with the radio jet, presenting broader velocity dispersions along the jet direction. Line-ratio maps indicate that the dominant excitation mechanism is photoionisation by the central AGN, with mechanical heating from shocks becoming important only at higher Eddington ratios. Across the sample, we detect coronal lines in all LLAGN when using nuclear-aperture spectra, and we can obtain spatially resolved kinematics only in NGC5128 and NGC4395, the highest Eddington ratio sources.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.06667-b31b1b.svg)](https://arxiv.org/abs/2611.06667) | **Euclid Quick Data Release (Q1). Refining the strong lens sample with Space Warps crowd-sourced science**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *16 pages, 9 figures, submitted to A&A*|
|**Abstract**|            We demonstrate the efficacy of crowd-sourced refinement of strong gravitational lens candidate samples by citizen scientists. Binary (lens/non-lens) classification by citizen scientists has been conducted previously, and here we extend such categorisation into `definite', `probable', `possible', and `non-lenses'. This grading is a required stage ahead of any strong lens sample publication and is usually carried out by (small) teams of researchers. We compare the responses of the volunteers to those of strong lens researchers and discuss the optimal strategy for lens classification and grading in future strong lens searches. Volunteers are found to be able to correctly identify systems of each grade, and their responses can be calibrated to be comparable to those of researchers. Grade posteriors, defined as the probability that a strong lens system classified by the volunteers is a certain grade, converge most rapidly for non-lens systems. We find that one researcher classification is equivalent to only 1.9 volunteer classifications when grading such systems. There is significant disagreement in the grading of intermediate-grade lens candidates amongst both volunteers and researchers. Moving towards calibrated lens probabilities for future lens searches would help to account for such subjectivity. If the Q1 lens search was rerun with volunteers performing lens grading (and corresponding calibration), a completeness of 79% for Grades A+B (corresponding to definite + probable lenses respectively) could be achieved with 41% purity prior to researcher inspection - this would correspond to an expected 12500 true lenses (i.e. excluding contaminants) in Euclid DR1.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.06916-b31b1b.svg)](https://arxiv.org/abs/2611.06916) | **ReveaLLAGN 3: Revealing the Infrared Spectral Energy Distributions of Low Luminosity AGN**  |
|| D. Ohlson, et al. -- incl., <mark>A. Dumont</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| **|
|**Abstract**|            The spectral energy distributions of active galactic nuclei (AGN) provide valuable information on the emission mechanisms taking place in their inner regions. Here we use the James Webb Space Telescope to study the near and mid-infrared spectral energy distributions of seven nearby low-luminosity active galaxy nuclei from 1.7 to 24 $\mu m$. We cleanly separate the AGN spectrum by performing spatial fits of the two-dimensional images across all wavelengths. We then compare these clean unresolved spectra to aperture-based spectra and show that they effectively remove galaxy contamination. Comparing our infrared measurements to X-ray emission, we find that low-luminosity AGN have enhanced near-infrared emission relative to more luminous AGN, a trend that correlates with radio-loudness, while their mid-infrared-to-X-ray ratios are comparable. A majority of our targets show strong compact silicate emission suggesting the presence of warm dust on parsec scales in these systems. We also quantify the near- and mid-infrared power-law slopes and colors of low-luminosity AGN. In the near-infrared we find AGN of all luminosities have similar spectral slopes and colors, however in the mid-infrared, we find low-luminosity AGN typically have steeper slopes and redder colors than more luminous AGN. Both the strong silicate emission and steep mid-infrared slopes disagree with existing models for low-luminosity AGN spectral energy distributions.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.05438-b31b1b.svg)](https://arxiv.org/abs/2611.05438) | **HLA-Net: A Physics-Guided Skeleton-Based Framework for Automatic Gravitational Arc Detection in Astronomical Images**  |
|| <mark>D. S</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *16 pages, 2 figures, planning to submit: The Open Journal of Astrophysics*|
|**Abstract**|            Gravitational arcs produced by strong gravitational lensing provide a unique approach for investigating dark matter distribution, galaxy cluster mass reconstruction and the structure of the Universe. Automated detection remains challenging because gravitational arcs are rare, often faint and morphologically similar to elongated this http URL this paper, we propose \textbf{HLA-Net (Hybrid Lens Arc Network)}, an interpretable multi-stage framework for automatic gravitational arc detection that explicitly exploits both the geometric morphology and physical characteristics of gravitational lensing without requiring manually annotated training this http URL proposed framework was evaluated on astronomical FITS images containing complex galaxy cluster structures. Starting from 169 initial arc candidates extracted from ordered skeleton representations, the multi-stage verification pipeline reduced the search space to 83 geometrically consistent candidates, while robust circle fitting successfully modelled 113 candidate arc structures. The results demonstrate that HLA-Net effectively suppresses false-positive detections while preserving physically reasonable lens candidates through an interpretable and modular verification strategy.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

... (5 outputs left out)


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.06873-b31b1b.svg)](https://arxiv.org/abs/2611.06873) | **Tracking turbulence for wind-driven halo removal using MagAO-X pyramid wavefront sensor images**  |
|| J. K. Kueny, et al. -- incl., <mark>J. Li</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *59 pages, 11 figures, 6 tables; Accepted for publication in SPIE JATIS*|
|**Abstract**|            The wind-driven halo (WDH) is a persistent, low spatial frequency noise artifact that arises due to the servo-lag error inherent to all adaptive optics (AO) instruments. Spatial filtering may be employed to overcome this artifact, however, filtering out the WDH while simultaneously preserving signal from an extended astrophysical object of interest is exceptionally challenging. Additionally, since the WDH changes in intensity and position angle through an observation, data-driven algorithms (e.g., KLIP) that are commonly used to subtract the starlight need to be overly-aggressive to remove both the static and dynamic noise components. Since wavefront sensors (WFSs) continuously track the closed-loop residual wavefront error, WFS telemetry presents the ideal resource for combating this type of noise artifact through postprocessing. Using archival WFS telemetry from MagAO-X, which is the "extreme" AO instrument for the 6.5-meter Magellan-Clay telescope, we demonstrate a novel workflow for WDH reconstruction and removal in individual coronagraphic science images. MagAO-X is equipped with a pyramid WFS capable of recording wavefront telemetry at a high-cadence which is saved during data acquisition. Given this, we detail how our WFS data processing pipeline, windsoCC, cross-correlates the recorded closed-loop wavefront to measure the wind vectors of several turbulent layers of the atmosphere above Las Campanas Observatory. We then make use of the wind parameters learned through windsoCC to reconstruct the WDH footprint by leveraging a parametric model. Notably, we demonstrate a dramatic improvement in object recovery using on-sky MagAO-X images of the disk around HR 4796A at visible wavelengths.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.05482-b31b1b.svg)](https://arxiv.org/abs/2611.05482) | **Euclid. Predicting scaling relations for emission-line galaxies using simulated data**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *12 pages, 9 figures, 5 pages appendix, submitted for publication to A&A*|
|**Abstract**|            The Euclid spectroscopic surveys will substantially increase the number of observed emission-line galaxies. We assess the ability of the Euclid Wide Survey (EWS) and Euclid Deep Survey (EDS), at the final DR3 depth, to recover the star-forming main sequence (MS), dust-mass relation (DMR), and mass-metallicity relation (MZR) from emission-line measurements. We use MAMBO (mocks with abundance matching in Bologna) spectroscopic simulations processed with the official Euclid pipelines. The MS is traced over $0\leq z\leq 4$ using Pa$\beta$, H$\alpha$, and [OII] as SFR tracers. The shallow EWS flux limit biases the MS towards higher normalisation and a flatter slope by selecting the brightest, most strongly star-forming galaxies. The deeper EDS enables almost complete sampling with Pa$\beta$ and H$\alpha$ at log$_{10}(M_*/M_\odot)\geq10.3$ over $0\leq z\leq 1.88$, with mild biases at both redshift limits. The [OII]-based MS is increasingly biased towards higher SFRs at the highest redshifts. The DMR is traced through the Balmer decrement over $1.45\leq z\leq 1.88$ and $0.9\leq z\leq 1.88$ in the EWS and EDS, respectively. The derived colour excess agree well with the input values, although detecting the faint H$\beta$ line restricts the sample to less obscured galaxies at fixed $M_*$, particularly in the EWS; the DMR is better sampled in the EDS. Finally, a golden sample detected in multiple optical rest-frame emission lines at $1.49\leq z\leq 1.88$ recovers gas-phase metallicities with typical accuracy of 0.1 dex at log$_{10}(M_*/M_\odot)\geq10$, while lower-mass galaxies show biases up to 0.15 dex. Relative to H-band limited sample, this sample preferentially traces the upper MS and lower MZR. Despite the simplified simulations lacking effects such as mutual spectral contamination and detector persistence, these forecasts provide a benchmark for future Euclid spectroscopic analyses.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.05484-b31b1b.svg)](https://arxiv.org/abs/2611.05484) | **Euclid: Forecasting the number counts of bright line emitters in the Euclid Wide Survey and Euclid Deep fields**  |
|| E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *13 pages, 7 figures. Submitted to A&A*|
|**Abstract**|            The Euclid mission will revolutionize our understanding of the visible Universe by collecting images and spectra of millions of galaxies across cosmic time and environments. Here, we forecast the number counts of galaxies emitting [OII], H\b{eta}, [OIII], H{\alpha} and Pa\b{eta} lines in the Euclid Wide Survey (EWS) and Deep Survey (EDS). We compare predictions from different models, including semi-analytic models and empirical scaling relations. We then simulate spectra for galaxies in the MAMBO lightcone using FastSpec, designed to rapidly generate one-dimensional spectra as observed by Euclid. We process these simulated spectra with the official Euclid spectroscopic pipeline to recover redshifts and line fluxes and derive expected counts. Predictions for H{\alpha}+[NII] emitters are broadly consistent across mocks and with existing observations. In the EWS, relative RMS scatter among mock predictions for integrated counts is approximately 19% for H{\alpha}+[NII], 44% for [OII] and 41% for [OIII], reflecting greater model dependence for oxygen-line forecasts. After accounting for flux losses and redshift misidentification, corrected flux measurements recover most intrinsic line counts, especially in the EDS. From the MAMBO-based FastSpec simulations, we predict that Euclid will detect about 42 million H{\alpha} emitters in the EWS and 1.7 million in the EDS. The same simulations predict approximately 0.5 and 0.8 million [OII] and [OIII] emitters, respectively, in the EDS, and about 1.3 and 7.5 million in the EWS. These non-H{\alpha} forecasts remain subject to substantial model-dependent uncertainties, as shown by scatter among the different mocks. These samples will be instrumental in pushing large-scale structure and baryon acoustic oscillation studies beyond redshift z~2, extending Euclid cosmological and galaxy evolution studies beyond what is achievable with H{\alpha} alone.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.06304-b31b1b.svg)](https://arxiv.org/abs/2611.06304) | **Euclid: Survey mask effects on clustering of galaxy clusters**  |
|| A. Fumagalli, et al. -- incl., <mark>K. Jahnke</mark> |
|*Appeared on*| *2026-11-05*|
|*Comments*| *12 pages, 11 figures*|
|**Abstract**|            Wide-area cluster catalogues from Stage-IV surveys will substantially improve the precision of cluster clustering measurements, making the control of observational systematic effects essential for cosmological inference. In preparation for the first Euclid data release (DR1), we assess the impact of the survey mask and non-homogeneous sky coverage on the two-point correlation function of galaxy clusters and its covariance. Using DR1-like lightcone mocks, we confirm that the irregular footprint does not introduce appreciable changes in the two-point correlation function when the same mask is applied to the random catalogue. We also find that spatial inhomogeneities, implemented as variations of the cluster selection function in different patches of the sky, do not significantly bias the clustering amplitude, up to levels of variation well above those expected for DR1. When the degree of inhomogeneity is increased to the regime where the effect becomes significant, we show that its amplitude is related to the maximum variation of the selection function among different regions, rather than to the number of such regions. We further find that irregular mask geometry and non-uniform sky coverage both increase the clustering covariance, introducing small, nearly scale-independent offsets that can be captured by nuisance parameters in the covariance model. Our results consolidate the treatment of mask systematic errors for DR1 and inform the methodological refinements required for future \Euclid cluster cosmology.         |
|<p style="color:red"> **ERROR** </p>| <p style="color:red">latex error list index out of range</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.05496/./Figures/KINEMATIC_MAPS/M104_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M94_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M81_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC1052_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC4258_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC5128_pPXF_Stellar_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M87_Hydrogen_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M104_Hydrogen_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M94_Hydrogen_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/M81_Hydrogen_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC1052_Hydrogen_Kinematics_mdegree6.png', 'tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC4258_Hydrogen_Kinem

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand{\orcid}[1]{\protect\href{https://orcid.org/#1}{\protect\includegraphics[width=8pt]{orcid.png}}}$
$\newcommand{\mum}{ {\mu{\rm m}}}$</div>



<div id="title">

# ReveaLLAGN 2: Exploring the Nuclear Gas and Stellar Kinematics of Nearby LLAGN with JWST/NIRSpec IFU

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.05496-b31b1b.svg)](https://arxiv.org/abs/2611.05496)<mark>Appeared on: 2026-11-05</mark> -  _Accepted for publication in Astronomy & Astrophysics_

</div>
<div id="authors">

<mark>A. Dumont</mark>, et al. -- incl., <mark>N. Neumayer</mark>

</div>
<div id="abstract">

**Abstract:** $_ Context:_$ Low-luminosity active galactic nuclei (LLAGN) represent the most common mode of black hole accretion in the local Universe, yet their nuclear structure, stellar and gas kinematics remain poorly constrained at parsec scales. $_ Aims:_$ We investigate the nuclear regions of eight nearby representative LLAGN, the ReveaLLAGN sample: M87, M104, M94, M81, NGC 1052, NGC 4258, NGC 5128 and NGC 4395. Using the high spatial resolution of the Near Infrared Spectrograph (NIRSpec) integral field unit on the James Webb Space Telescope (JWST), we probe the stellar, molecular hydrogen, hydrogen recombination, and high ionization coronal lines kinematics and their spatial distribution, exploring how detectability and spatial distributions of these lines vary across different Eddington ratios. $_ Methods:_$ After cleaning  the JWST/NIRSpec data of point-spread-function sinusoidal artifacts, we spatially binned the data to increase its signal-to-noise. We fit the binned spectra using ${\sc pPXF}$ with stellar templates and several gas emission lines, including $H_2$ molecular lines, $H_2$ recombination lines (Pa $\alpha$ and Br $\gamma$ ), and coronal lines such as [ Si VI ] , [ Mg IV ] , and [ Ar VI ] , adopting a Gaussian profile to derive their kinematics. $_ Results:_$ The stellar line-of-sight velocity maps reveal a slowly rotating disk in the nuclei of all our targets, characterized by an anti-correlation with the $h_3$ Gauss–Hermite moment. The stellar velocity dispersion rises toward the central black hole. Only in M104 and M94 does the velocity dispersion display a more intricate pattern that departs from a simple disk configuration, indicating the presence of multiple kinematic components in these galaxies. Across the sample, the gas distribution and kinematics exhibit rotation that is frequently decoupled from the stellar rotation, and often visually aligned with the radio jet, presenting broader velocity dispersions along the jet direction. Line-ratio maps of $H_2$ 1–0 S(1) to Br $\gamma$ indicate that the dominant excitation mechanism is photoionization by the central AGN, with mechanical heating from shocks becoming important only at higher Eddington rations. $_ Conclusions:_$ Across the ReveaLLAGN sample, we observe slow, disk-like stellar rotation that is decoupled from the gas kinematics, suggesting that the stellar gravitational potential is largely decoupled from the current accretion state. The gas kinematics are more diverse across the targets, with several sources showing kinematic features spatially associated with the radio jets. Across the sample, we detect coronal lines in all LLAGN when using nuclear-aperture spectra. However, no major dependence on Eddington ratio is found, except in the brightness of the coronal lines, which are sufficiently bright to obtain spatially resolved kinematics only in the highest-Eddington-ratio sources, NGC 5128 and NGC 4395.

</div>

<div id="div_fig1">

<img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M104_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.1" width="16%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M94_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.2" width="16%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M81_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.3" width="16%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC1052_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.4" width="16%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC4258_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.5" width="16%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC5128_pPXF_Stellar_Kinematics_mdegree6.png" alt="Fig3.6" width="16%"/>

**Figure 3. -** ReveaLLAGN stellar kinematic maps for the first four velocity moments: velocity, velocity dispersion, $h_3$, and $h_4$. The maps are oriented north up and east to the left. The dashed black line shows the best fit stellar kinematic PA.  (*fig:stellar_kinematic_maps*)

</div>
<div id="div_fig2">

<img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M87_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.1" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M104_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.2" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M94_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.3" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/M81_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.4" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC1052_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.5" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC4258_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.6" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC5128_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.7" width="12%"/><img src="tmp_2611.05496/./Figures/KINEMATIC_MAPS/NGC4395_Hydrogen_Kinematics_mdegree6.png" alt="Fig4.8" width="12%"/>

**Figure 4. -** kinematics maps of the molecular Hydrogen $H_2$ and Hydrogen recombination lines Pa$\alpha$ and Br$\gamma$. The first three panels show the LOS velocity and the last two the velocity dispersion. The dashed black line shows the best-fit gas kinematic PA. The solid white line marks the direction of the radio jet. **The dashed fuchsia line shows the stellar kinematic PA.** (*fig:gas_kinematic_maps*)

</div>
<div id="div_fig3">

<img src="tmp_2611.05496/./Figures/Flux_Maps/M87_pPXF_flux_maps_mdegree6.png" alt="Fig5.1" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/M104_pPXF_flux_maps_mdegree6.png" alt="Fig5.2" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/M94_pPXF_flux_maps_mdegree6.png" alt="Fig5.3" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/M81_pPXF_flux_maps_mdegree6.png" alt="Fig5.4" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/NGC1052_pPXF_flux_maps_mdegree6.png" alt="Fig5.5" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/NGC4258_pPXF_flux_maps_mdegree6.png" alt="Fig5.6" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/NGC5128_pPXF_flux_maps_mdegree6.png" alt="Fig5.7" width="12%"/><img src="tmp_2611.05496/./Figures/Flux_Maps/NGC4395_pPXF_flux_maps_mdegree6.png" alt="Fig5.8" width="12%"/>

**Figure 5. -** Flux and line ratio maps. The left three panels show the flux maps for the $H_2$ 1-0 S(1),$Pa\alpha$, and Br$\gamma$ lines. The right two panels show the $H_2$/Br$\gamma$ line ratio map and a colorized version highlighting the different excitation mechanism. Values of $H_2$/Br$\gamma \leq 0.4$(blue) indicate star forming regions,  $0.4<$ $H_2$/Br$\gamma \leq 2.0$ low-AGN excitation (orange), $2.0<$ $H_2$/Br$\gamma \leq 6.0$ high-AGN and $H_2$/Br$\gamma$ above 6.0 shock excitation (green). The white line shows the direction of the radio jet. (*fig:flux_maps*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.05496"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\tinier}{\fontsize{7.5pt}{7.5pt}\selectfont}$
$\newcommand{\ignoreme}[1]\usepackage$
$\newcommand{\mytexttilde}{\raisebox{0.5ex}{\texttildelow}}$
$\newcommand{\orcid}[1]{\orcidlink{#1}}$</div>



<div id="title">

# Euclid Quick Data Release (Q1): Refining the strong lens sample with Space Warps crowd-sourced science$\thanks{This publication has been made possible by the participation of volunteers in the Space Warps project. Their contributions are individually acknowledgedin Appendix \ref{Ap: SW Volunteers} and on the Space Warps \href{https://www.zooniverse.org/projects/aprajita/space-warps-esa-euclid/about/team}{`The Team'} webpage.}$

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.06667-b31b1b.svg)](https://arxiv.org/abs/2611.06667)<mark>Appeared on: 2026-11-05</mark> -  _16 pages, 9 figures, submitted to A&A_

</div>
<div id="authors">

E. Collaboration, et al. -- incl., <mark>K. Jahnke</mark>

</div>
<div id="abstract">

**Abstract:** We demonstrate the efficacy of crowd-sourced refinement of strong gravitational lens candidate samples by citizen scientists. Binary (lens/non-lens) classification by citizen scientists has been conducted previously, and here we extend such categorisation into `definite', `probable', `possible', and `non-lenses'. This grading is a required stage ahead of any strong lens sample publication and is usually carried out by (small) teams of researchers.We compare the responses of the volunteers to those of strong lens researchers and discuss the optimal strategy for lens classification and grading in future strong lens searches. Volunteers are found to be able to correctly identify systems of each grade, and their responses can be calibrated to be comparable to those of researchers.Grade posteriors, defined as the probability that a strong lens system classified by the volunteers is a certain grade, converge most rapidly for non-lens systems. We find that one researcher classification is equivalent to only $\rt{1.9}$ volunteer classifications when grading such systems.There is significant disagreement in the grading of intermediate-grade lens candidates amongst both volunteers and researchers. Moving towards calibrated lens probabilities for future lens searches would help to account for such subjectivity. If the Q1 lens search was rerun with volunteers performing lens grading (and corresponding calibration), a completeness of $79\%$ for Grades A+B (corresponding to definite + probable lenses respectively) could be achieved with $41\%$ purity prior to researcher inspection -- this would correspond to an expected $12 500$ true lenses (i.e. excluding contaminants) in Euclid DR1.

</div>

<div id="div_fig1">

<img src="tmp_2611.06667/./Images/Random_Grade_examples.png" alt="Fig1" width="100%"/>

**Figure 1. -** Random selection of systems shown to both researcher and volunteer graders, along with the corresponding number of grades received from each. The cutouts are highlighted by their consensus researcher grade: A; green, B; orange, C; red, X; black. (*Random_Grade_Examples*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.06667"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand\latex{La\TeX}$
$\newcommand{\sersic}{Sérsic}$
$\newcommand{\thebibliography}{\DeclareRobustCommand{\VAN}[3]{##3}\VANthebibliography}$</div>



<div id="title">

# ReveaLLAGN 3: Revealing the Infrared Spectral Energy Distributions of Low Luminosity AGN

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.06916-b31b1b.svg)](https://arxiv.org/abs/2611.06916)<mark>Appeared on: 2026-11-05</mark> - 

</div>
<div id="authors">

D. Ohlson, et al. -- incl., <mark>A. Dumont</mark>

</div>
<div id="abstract">

**Abstract:** The spectral energy distributions of active galactic nuclei (AGN) provide valuable information on the emission mechanisms taking place in their inner regions. Here we use the James Webb Space Telescope to study the near and mid-infrared spectral energy distributions of seven nearby low-luminosity active galaxy nuclei from 1.7 to 24 $\mu$ m. We cleanly separate the AGN spectrum by performing spatial fits of the two-dimensional images across all wavelengths. We then compare these clean unresolved spectra to aperture-based spectra and show that they effectively remove galaxy contamination. Comparing our infrared measurements to X-ray emission, we find that low-luminosity AGN have enhanced near-infrared emission relative to more luminous AGN, a trend that correlates with radio-loudness, while their mid-infrared-to-X-ray ratios are comparable. A majority of our targets show strong compact silicate emission suggesting the presence of warm dust on parsec scales in these systems.We also quantify the near- and mid-infrared power-law slopes and colors of low-luminosity AGN. In the near-infrared we find AGN of all luminosities have similar spectral slopes and colors, however in the mid-infrared, we find low-luminosity AGN typically have steeper slopes and redder colors than more luminous AGN. Both the strong silicate emission and steep mid-infrared slopes disagree with existing models for low-luminosity AGN spectral energy distributions.

</div>

<div id="div_fig1">

<img src="tmp_2611.06916/./spec_compare_lum.png" alt="Fig11" width="100%"/>

**Figure 11. -** Nuclear Spectra from the ReveaLLAGN targets, comparing the results of our decomposed point-source AGN against those from a background-subtracted nuclear aperture.
    *Top* -- Eddington-normalized luminosity of the unresolved AGN components, resulting from fitting and extraction methods described in \S\ref{sec:fit}.
    *Middle* -- Unresolved AGN component flux density within 1.7" aperture.
    *Bottom* -- Nuclear spectra extracted using the methods of [Goold, et. al (2026)](https://ui.adsabs.harvard.edu/abs/2026arXiv260116977G). Despite background subtraction and aperture correction, contamination from surrounding host-galaxy emission is still apparent throughout the bottom panel, most notably at the CO bandhead absorption at $\rm \sim2.3 \mu m$ visible in Sombrero, M94, NGC4258, and M81 and also shortwards of 5 $\mu$m where the slope flattens relative to our decomposed AGN spectra, especially in M94 and Sombrero. Contamination from the nuclear host-galaxy flux in aperture-extracted spectra shows the importance of decoupling the AGN and galaxy components to extract accurate LLAGN continua. We include an average LLAGN IR spectrum template in black, made using our decomposed AGN spectra. Visible jumps in flux throughout the spectrum of M94 result from the separate models used to account for extreme PAH emission, represented in the top two panels by a more transparent line and (*fig:spectra*)

</div>
<div id="div_fig2">

<img src="tmp_2611.06916/./galaxy_spectra.png" alt="Fig2" width="100%"/>

**Figure 2. -** Host galaxy component spectra in the 1.7$"$ nuclear region. CO bandheads are clearly visible for all included targets at $\sim$2.3 $\mu$m. PAH features are well modeled for multiple galaxies, most notably M94, which necessitate special fitting methods as described in \S\ref{subsec:specialgalaxies}. We exclude NGC 4395 from our figure because its simplified galaxy emission requires fewer functions and is nonexistent in some channels. (*fig:galspec*)

</div>
<div id="div_fig3">

<img src="tmp_2611.06916/./nir_xr_eddington.png" alt="Fig3" width="100%"/>

**Figure 3. -** Ratio of integrated 2.2 $\mu$m and 2–10 keV X-ray luminosities vs Eddington ratio. Small points show data from [Dumont, et. al (2020)](https://ui.adsabs.harvard.edu/abs/2020ApJ...888...19D)(diamonds), [Burtscher, et. al (2015)](https://ui.adsabs.harvard.edu/abs/2015A&A...578A..47B)(squares), and [M\"uller-S\'anchez, et. al (2018)](https://ui.adsabs.harvard.edu/abs/2018ApJ...858...48M)(circles). Eddington ratios for comparison targets are calculated using black hole masses from [Bi, Feng and Ho (2020)](https://ui.adsabs.harvard.edu/abs/2020ApJ...900..124B). The dashed horizontal green line shows the mean flux ratio for all data points where log($\rm L_{Bol}/L_{Edd}$)$\leq$-3, and the red line for those $>$-3. Shaded regions show $1\sigma$ errors calculated with bootstrapping. Our measurements support previous results suggesting a NIR flux excess for LLAGN. (*fig:nirxr*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.06916"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

136  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

10  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

6  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
